In [81]:
import random
import math
from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

In [82]:
a = Value(-4.0)
b = Value(2.0)
c = a + b

print(f"Forward Pass: {c.data:.4f}")

c.backward()

print(f"dc/da: {a.grad:.4f}")
print(f"dc/db: {b.grad:.4f}")

Forward Pass: -2.0000
dc/da: 1.0000
dc/db: 1.0000


In [83]:
a = Value(-4.0)
b = Value(2.0)
d = a + (b**2)

print(f"Forward Pass: {d.data:.4f}")

d.backward()

print(f"dd/da: {a.grad:.4f}")
print(f"dd/db: {b.grad:.4f}")

Forward Pass: 0.0000
dd/da: 1.0000
dd/db: 4.0000


In [84]:
a = Value(-4.0000001)
b = Value(1.999999)
d = a + (b**2)

print(f"Forward Pass: {d.data:.10f}")

d.backward()

print(f"dd/da: {a.grad:.4f}")
print(f"dd/db: {b.grad:.4f}")

Forward Pass: -0.0000041000
dd/da: 1.0000
dd/db: 4.0000


In [85]:
# Find minima of a function by using recursive gradient descent

NUM_EPOCHS = 10000
DIFF = 0.001
FN = lambda x, y: 2*((x**2) + (y**2) + 1)

a = Value(-4.1)
b = Value(2.3)
f = FN(a, b)

print("BEFORE:")
print(f"Initial value of a: {a.data:.10f}")
print(f"Initial value of b: {b.data:.10f}")
print(f"Initial value of f: {f.data:.10f}")

i = 0
while i < NUM_EPOCHS:
    f.backward()
    a = Value(a.data - (a.grad * DIFF))
    b = Value(b.data - (b.grad * DIFF))
    f = FN(a, b)
    i += 1

print("\nAFTER:")
print(f"Final value of a: {a.data:.10f}")
print(f"Final value of b: {b.data:.10f}")
print(f"Final (minimized) value of f: {f.data:.10f}")

BEFORE:
Initial value of a: -4.1000000000
Initial value of b: 2.3000000000
Initial value of f: 46.2000000000

AFTER:
Final value of a: -0.0000000000
Final value of b: 0.0000000000
Final (minimized) value of f: 2.0000000000


In [86]:
# Build and fit a simple predictive model to synthetic data using micrograd

# Generate data: train and test
dataset_size = 1000
train_set_size = math.floor(dataset_size * 0.8)
real_pattern = lambda x, y: x**2 + y

dataset = []

for d in range(dataset_size):
    x = random.randint(-500, 500)
    y = random.randint(-500, 500)
    target = real_pattern(x, y)
    dataset.append((x, y, target))

# After generating dataset
x_vals = [d[0] for d in dataset]
y_vals = [d[1] for d in dataset]
targets = [d[2] for d in dataset]

# Calculate mean and std
x_mean = sum(x_vals) / len(x_vals)
y_mean = sum(y_vals) / len(y_vals)
t_mean = sum(targets) / len(targets)

x_std = (sum((x - x_mean)**2 for x in x_vals) / len(x_vals))**0.5
y_std = (sum((y - y_mean)**2 for y in y_vals) / len(y_vals))**0.5
t_std = (sum((t - t_mean)**2 for t in targets) / len(targets))**0.5

# Normalize
dataset = [((x - x_mean) / x_std, 
            (y - y_mean) / y_std, 
            (t - t_mean) / t_std) 
           for x, y, t in dataset]
normalized_inputs = [((x - x_mean) / x_std, (y - y_mean) / y_std) for x, y, _ in dataset]
normalized_outputs = [(t - t_mean) / t_std for _, _, t in dataset]

# Construct train & test sets
train_set = dataset[0: train_set_size]
test_set = dataset[train_set_size:]

train_set_inputs = normalized_inputs[0: train_set_size]
train_set_outputs = normalized_outputs[0: train_set_size]

test_set_inputs = normalized_inputs[train_set_size:]
test_set_outputs = normalized_outputs[train_set_size:]

In [145]:
# Build neural network model
n = MLP(2, [2, 1])

In [151]:
# Train neural network
EPOCHS = 100
LEARNING_RATE = 0.00005

for e in range(EPOCHS):
    # forward pass
    preds = [n(inp) for inp in train_set_inputs]
    
    # compute loss
    loss = sum((pred - y_real)**2 for pred, y_real in zip(preds, train_set_outputs))**0.5
    print(f"{e} {loss.data:.6f}")
    
    # zero_grad
    n.zero_grad()
    
    # backprop
    loss.backward()
    
    # adjust weights
    for p in n.parameters():
        p.data += -LEARNING_RATE*p.grad

print("\n TRAINING COMPLETE")
loss = sum((pred - y_real)**2 for pred, y_real in zip(preds, train_set_outputs))
print(f"Final Loss: {loss.data:.6f}")

0 0.101470
1 0.059343
2 0.051995
3 0.051846
4 0.051832
5 0.051830
6 0.051829
7 0.051829
8 0.051828
9 0.051828
10 0.051827
11 0.051827
12 0.051826
13 0.051826
14 0.051825
15 0.051825
16 0.051824
17 0.051824
18 0.051824
19 0.051823
20 0.051823
21 0.051822
22 0.051822
23 0.051821
24 0.051821
25 0.051820
26 0.051820
27 0.051819
28 0.051819
29 0.051818
30 0.051818
31 0.051817
32 0.051817
33 0.051816
34 0.051816
35 0.051815
36 0.051815
37 0.051814
38 0.051814
39 0.051813
40 0.051813
41 0.051812
42 0.051812
43 0.051811
44 0.051811
45 0.051810
46 0.051810
47 0.051809
48 0.051809
49 0.051808
50 0.051808
51 0.051808
52 0.051807
53 0.051807
54 0.051806
55 0.051806
56 0.051805
57 0.051805
58 0.051804
59 0.051804
60 0.051803
61 0.051803
62 0.051802
63 0.051802
64 0.051801
65 0.051801
66 0.051800
67 0.051800
68 0.051799
69 0.051799
70 0.051798
71 0.051798
72 0.051797
73 0.051797
74 0.051796
75 0.051796
76 0.051795
77 0.051795
78 0.051794
79 0.051794
80 0.051793
81 0.051793
82 0.051793
83 0.051792
84

In [105]:
# Build simple model (using basic quadratic math function); choose hyperparameters and loss fn
model_template = lambda w: (lambda x, y: w[0]*(x**2) + w[1]*(x) + w[2]*(y) + w[3])
# where: x, y are inputs and w is the list of weights

# Initializing weights with random values
w = [Value(10.0), Value(10.0), Value(-10.0), Value(10.0)]

NUM_EPOCHS = 1500
STEP_SIZE = 0.001

# Run train loop
for i in range(NUM_EPOCHS):
    model = model_template(w)
    loss_fn = Value(0.0)
    
    for data in train_set:
        pred = model(data[0], data[1])
        loss_fn += ((pred - data[2])**2)

    loss_fn = loss_fn**0.5

    if i % 100 == 0:
        print(f"loss value at epoch {i} = {loss_fn.data:.5f}")

    loss_fn.backward()
    
    # Update weights and zero gradients
    for weight in w:
        weight.data -= weight.grad * STEP_SIZE
        weight.grad = 0  # Important!

print("\n\n TRAINING COMPLETE: ")
for j in range(len(w)):
    print(f"w[{j}] = {w[j].data:.9f}")

# Validate using test data

loss value at epoch 0 = 710.34687
loss value at epoch 100 = 561.29565
loss value at epoch 200 = 425.02681
loss value at epoch 300 = 305.21621
loss value at epoch 400 = 204.01140
loss value at epoch 500 = 119.30488
loss value at epoch 600 = 48.08225
loss value at epoch 700 = 7.15188
loss value at epoch 800 = 1.02755
loss value at epoch 900 = 1.02755
loss value at epoch 1000 = 1.02755
loss value at epoch 1100 = 1.02755
loss value at epoch 1200 = 1.02755
loss value at epoch 1300 = 1.02755
loss value at epoch 1400 = 1.02755


 TRAINING COMPLETE: 
w[0] = 1.150326121
w[1] = 0.014843428
w[2] = 0.004994648
w[3] = -1.118392220


In [79]:
# Validate using test data
trained_model = model_template(w)

loss_fn = Value(0.0)
for data in test_set:
    pred = trained_model(data[0], data[1])
    loss_fn += ((pred - data[2])**2)
loss_fn = loss_fn**0.5

print(f"Test set loss = {loss_fn.data:.9f}")

Test set loss = 0.457460370
